In [ ]:
from IPython.core.display import HTML
with open('../style.css', 'r') as file:
    css = file.read()
HTML(css)

# Drawing Finite State Machines

This notebook provides functions that convert finite state machines into strings and into graphs.

## Type Checking

The functions in this notebook carry *type annotations*.  *Python* itself ignores these annotations, but the
type checker [*basedpyright*](https://docs.basedpyright.com) can use them to find errors before the program is
run.  In *JupyterLab*, the extension *jupyterlab-lsp* runs *basedpyright* in the background and underlines
type errors while you type.  On the command line, the command
```
basedpyright FSM-2-Dot.ipynb
```
checks the whole notebook.  The settings of the type checker are stored in the file `pyrightconfig.json` in the
directory `Python`.

Both packages, `basedpyright` and `jupyterlab-lsp`, are installed by the script `fl.sh`.  Start `jupyter lab` in the directory `Python`, so that the settings in `pyrightconfig.json` are used.

The functions in this notebook work both for deterministic and for non-deterministic finite state machines.
The type `FSM` describes both kinds of machines: the states can be arbitrary values, and the transition
function maps pairs of a state and a character either to a state (for a <span style="font-variant:small-caps;">Dfa</span>)
or to a set of states (for an <span style="font-variant:small-caps;">Nfa</span>).

In [ ]:
from typing import Any

type FSM = tuple[set[Any] | frozenset[Any], set[str], dict[tuple[Any, str], Any], Any, set[Any] | frozenset[Any]]

The function `dfa2string` converts the given <span style="font-variant:small-caps;">Dfa</span> into a string.

**Function `dfa2string(dfa)`**
- *Input:* `dfa` is a <span style="font-variant:small-caps;">Dfa</span>.
- *Output:* A string that describes `dfa`.  The states are renamed to `S0`, `S1`, ... .

In [ ]:
def dfa2string(dfa: FSM) -> str:
    states, sigma, delta, q0, final = dfa
    result = ''
    n = 0
    statesToNames: dict[Any, str] = {}
    for q in states:
        statesToNames[q] = f'S{n}'
        n += 1
    result += 'states: {S0, ..., ' + f'S{n-1}' + '}\n\n'   
    result += f'start state: {statesToNames[q0]}' + '\n\n'
    result += 'state encoding:\n'
    for q in states:
        result += f'{statesToNames[q]} = {q}' + '\n'
    result += '\ntransitions:\n'
    for q in states:
        for c in sigma: 
            if delta.get((q, c)) != None:
                result += f'delta({statesToNames[q]}, {c}) = {statesToNames[delta[(q, c)]]}' + '\n'
    result += '\nset of accepting states: {'
    result += ', '.join({ statesToNames[q] for q in final })
    result += '}\n'
    return result

In [ ]:
import graphviz as gv

The function `dfa2dot` converts the given <span style="font-variant:small-caps;">Dfa</span> into a graph in dot-format.

**Function `dfa2dot(dfa)`**
- *Input:* `dfa` is a <span style="font-variant:small-caps;">Dfa</span>.
- *Output:* A pair consisting of a `graphviz` graph of `dfa` and a dictionary that maps the states of `dfa` to their names in the graph.

In [ ]:
def dfa2dot(dfa: FSM) -> tuple[gv.Digraph, dict[Any, str]]:
    states, sigma, delta, q0, final = dfa
    dot = gv.Digraph('Deterministic FSM')
    dot.graph_attr['rankdir'] = 'LR'
    n = 0              # used to assign names to states
    statesToNames: dict[Any, str] = {} # assigns a name to every state
    for q in states:
        statesToNames[q] = f'S{n}'
        n += 1
    startName = statesToNames[q0]
    dot.node('1', label='', width='0.1', height='0.1', style='filled', color='blue')
    dot.edge('1', startName)
    for q in states:
        if q in final:
            dot.node(statesToNames[q], peripheries='2')
        else:
            dot.node(statesToNames[q])
    for q in states:
        for c in sigma:
            p = delta.get((q, c))
            if p != None:
                dot.edge(statesToNames[q], statesToNames[p], label = c)
    return dot, statesToNames

The function `nfa2string` converts a non-deterministic finite state machine `nfa` into a string.

**Function `nfa2string(nfa)`**
- *Input:* `nfa` is an <span style="font-variant:small-caps;">Nfa</span>.
- *Output:* A string that describes `nfa`.

In [ ]:
def nfa2string(nfa: FSM) -> str:
    states, sigma, delta, q0, final = nfa
    n       = 0
    result  = ''
    result += f'states: {states}' + '\n\n'   
    result += f'start state: {q0}' + '\n\n'
    result += 'transitions:\n'
    for q in states:
        for c in sigma:
            S = delta.get((q, c))
            if S != None:
                for p in S:
                    result += f'[{q}, {c}] |-> {p}' + '\n'
        S = delta.get((q, '𝜀'))
        if S != None:
            for p in S:
                result += f'[{q}, 𝜀] |-> {p}' + '\n'
    result += '\n' + f'set of accepting states: {final}' + '\n'
    return result

The function `nfa2dot` takes a non-deterministic finite state machine and converts it 
into a a dot graph.

**Function `nfa2dot(nfa)`**
- *Input:* `nfa` is an <span style="font-variant:small-caps;">Nfa</span>.
- *Output:* A `graphviz` graph of `nfa`.

In [ ]:
def nfa2dot(nfa: FSM) -> gv.Digraph:
    states, sigma, delta, q0, final = nfa
    result = ''
    n      = 0
    startName = str(q0)
    dot = gv.Digraph('Non-Deterministic FSM')
    dot.graph_attr['rankdir'] = 'LR'
    dot.node('0', label='', width='0.1', height='0.1', style='filled', color='blue')
    dot.edge('0', startName)
    for q in states:
        if q in final:
            dot.node(str(q), peripheries='2')
        else:
            dot.node(str(q))
    for q in states:
        S = delta.get((q, '𝜀'))
        if S != None:
            for p in S:
                dot.edge(str(q), str(p), label='𝜀', weight='0.1')
    for q in states:
        for c in sigma:
            S = delta.get((q, c))
            if S != None:
                for p in S:
                    dot.edge(str(q), str(p), label=c, weight='10')
    return dot